In [1]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

sys.path.append(os.path.join(os.getcwd(), '../src'))
from utils import load_corpus_data
from faiss_encoder import FaissEncoder
from metrics import calculate_recall_at_k

/home/fernandogd/Documents/Investigacion/Transformers/Repositories/HERBERT/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CORPORA = ["DisTEMIST", "MedProcNER", "SympTEMIST"]
F_TYPE = "FlatIP"
MAX_LENGTH = 256
K_VALUES = [1, 5, 25, 200]
BASE_MODELS = {
    "DisTEMIST": "ICB-UMA/DisTEMIST-bi-encoder",
    "MedProcNER": "ICB-UMA/MedProcNER-bi-encoder",
    "SympTEMIST": "ICB-UMA/SympTEMIST-bi-encoder",
}
MODEL_LIST = ["ICB-UMA/HERBERT-GP", "ICB-UMA/HERBERT-GP-30", "ICB-UMA/HERBERT-P", "ICB-UMA/HERBERT-P-30", "ICB-UMA/ClinLinker", "cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR", "cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR-large","PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"]
load_dotenv("../.env")
DATA_PATH = os.getenv("DATA_PATH")    
OUTPUT_PATH = "../output/"

In [3]:
def model_evaluation(
    hf_model: str,
    train_gaz_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_df: pd.DataFrame,
    gaz_df: pd.DataFrame,
    k_values: list[int],
) -> tuple[dict, dict, dict]:
    
    faiss_encoder = FaissEncoder(hf_model, F_TYPE, MAX_LENGTH, train_gaz_df)
    faiss_encoder.fit_faiss()

    candidates, codes, _ = faiss_encoder.get_candidates(
        test_df["term"].tolist(),
        k=300,
    )

    preds = test_df.copy()
    preds["candidates"] = candidates
    preds["codes"] = codes

    um_preds, uc_preds = split_um_uc_filtered(
        preds,
        train_df,
        gaz_df,
    )

    gs_metrics = calculate_recall_at_k(preds, k_values)
    um_metrics = calculate_recall_at_k(um_preds, k_values)
    uc_metrics = calculate_recall_at_k(uc_preds, k_values)

    return gs_metrics, um_metrics, uc_metrics

def split_um_uc_filtered(
    df: pd.DataFrame,
    train_df: pd.DataFrame,
    gaz_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    um_df = df[
        ~df["term"].isin(train_df["term"])
        & ~df["term"].isin(gaz_df["term"])
    ]
    uc_df = df[
        ~df["code"].isin(train_df["code"])
    ]

    um_filtered_df = um_df[
        ~um_df["code"].str.contains(r"\+|NO_CODE|NO_MAP", na=False)
    ]
    uc_filtered_df = uc_df[
        ~uc_df["code"].str.contains(r"\+|NO_CODE|NO_MAP", na=False)
    ]
    return um_filtered_df, uc_filtered_df

def save_results_tsv(
    results: dict,
    out_path: str,
    filename: str,
) -> None:
    (
        pd.DataFrame.from_dict(results, orient="index")
        .reset_index()
        .rename(columns={"index": "name"})
        .rename(columns=lambda c: f"Recall@{c}" if isinstance(c, int) else c)
        .sort_values("name")
        .to_csv(os.path.join(out_path, filename), sep="\t", index=False)
    )

In [4]:
for corpus in CORPORA:
    print(f"\n{'='*80}\n[START] Corpus: {corpus}\n{'='*80}")

    test_df, train_df, gaz_df = load_corpus_data(DATA_PATH, corpus)
    train_gaz_df = pd.concat([train_df[["term", "code"]], gaz_df[["term","code"]]], ignore_index=True) 
    train_gaz_df.drop_duplicates(inplace=True)
    aux_path = os.path.join(OUTPUT_PATH, corpus)
    os.makedirs(aux_path, exist_ok=True)

    gs_results, um_results, uc_results = {}, {}, {}

    base_model = BASE_MODELS.get(corpus)
    if base_model is not None:
        print(f"Evaluating corpus-specific model: {base_model}")
        name = base_model.split("/")[-1]
        gs_m, um_m, uc_m = model_evaluation(
            base_model,
            train_gaz_df,
            test_df,
            train_df,
            gaz_df,
            K_VALUES,
        )
        gs_results[name], um_results[name], uc_results[name] = gs_m, um_m, uc_m

    # 2) Resto de modelos
    print("Evaluating remaining models...")
    for i, model in enumerate(MODEL_LIST, 1):
        print(f"({i}/{len(MODEL_LIST)}) {model}")
        name = model.split("/")[-1]
        gs_m, um_m, uc_m = model_evaluation(
            model,
            train_gaz_df,
            test_df,
            train_df,
            gaz_df,
            K_VALUES,
        )
        gs_results[name], um_results[name], uc_results[name] = gs_m, um_m, uc_m

    save_results_tsv(gs_results, aux_path, "results_gs.tsv")
    save_results_tsv(um_results, aux_path, "results_um.tsv")
    save_results_tsv(uc_results, aux_path, "results_uc.tsv")




[START] Corpus: DisTEMIST
Evaluating corpus-specific model: ICB-UMA/DisTEMIST-bi-encoder
Evaluating remaining models...
(1/8) ICB-UMA/HERBERT-GP
(2/8) ICB-UMA/HERBERT-GP-30
(3/8) ICB-UMA/HERBERT-P
(4/8) ICB-UMA/HERBERT-P-30
(5/8) ICB-UMA/ClinLinker
(6/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR
(7/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR-large
(8/8) PlanTL-GOB-ES/roberta-base-biomedical-clinical-es


Some weights of RobertaModel were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-biomedical-clinical-es and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[START] Corpus: MedProcNER
Evaluating corpus-specific model: ICB-UMA/MedProcNER-bi-encoder
Evaluating remaining models...
(1/8) ICB-UMA/HERBERT-GP
(2/8) ICB-UMA/HERBERT-GP-30
(3/8) ICB-UMA/HERBERT-P
(4/8) ICB-UMA/HERBERT-P-30
(5/8) ICB-UMA/ClinLinker
(6/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR
(7/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR-large
(8/8) PlanTL-GOB-ES/roberta-base-biomedical-clinical-es


Some weights of RobertaModel were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-biomedical-clinical-es and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[START] Corpus: SympTEMIST
Evaluating corpus-specific model: ICB-UMA/SympTEMIST-bi-encoder
Evaluating remaining models...
(1/8) ICB-UMA/HERBERT-GP
(2/8) ICB-UMA/HERBERT-GP-30
(3/8) ICB-UMA/HERBERT-P
(4/8) ICB-UMA/HERBERT-P-30
(5/8) ICB-UMA/ClinLinker
(6/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR
(7/8) cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR-large
(8/8) PlanTL-GOB-ES/roberta-base-biomedical-clinical-es


Some weights of RobertaModel were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-biomedical-clinical-es and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
